В чистом inference-ноутбуке выполняется загрузка production-версии модели и тестовый uplift-predict на входном payload. Это подтверждает, что финальная модель может быть не только обучена и оценена, но и воспроизводимо использована как inference-компонент.

In [ ]:
import os
import sys
import json
import numpy as np
import pandas as pd
import joblib

current_dir = os.getcwd()
if current_dir not in sys.path:
    sys.path.insert(0, current_dir)

parent_dir = os.path.dirname(current_dir)
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

import mlflow
import mlflow.catboost
import mlflow.sklearn

from helpers.feature_extraction_expanded import UpliftFeatureExtractorExpanded
from helpers.clusterization_features_extraction import UmapClusterTransformer

# Конфигурация окружения для подключения к S3
os.environ["MLFLOW_TRACKING_URI"] = "http://localhost:5000"
os.environ["MLFLOW_S3_ENDPOINT_URL"] = "http://localhost:9000"
os.environ["AWS_ACCESS_KEY_ID"] = "minioadmin"
os.environ["AWS_SECRET_ACCESS_KEY"] = "minioadmin"
os.environ["MLFLOW_S3_IGNORE_TLS"] = "true"

class S3XLearnerPredictor:
    def __init__(self, model_tau_0, model_tau_1, model_propensity):
        self.model_tau_0 = model_tau_0
        self.model_tau_1 = model_tau_1
        self.model_propensity = model_propensity

    def predict_uplift(self, X: pd.DataFrame) -> np.ndarray:
        # Предсказание эффектов по веткам
        tau0 = self.model_tau_0.predict(X)
        tau1 = self.model_tau_1.predict(X)

        # Оценка propensity score
        g = self.model_propensity.predict_proba(X)[:, 1]
        uplift = g * tau0 + (1 - g) * tau1
        
        return uplift

def load_models_from_s3_registry(version: int = 2):
    print("\nПодключение к S3 и загрузка компонентов модели")
    
    # Загружаем CatBoost регрессоры
    model_tau_0_uri = f"models:/model_tau_0/{version}"
    print(f"Загрузка model_tau_0 из реестра: {model_tau_0_uri}")
    model_tau_0 = mlflow.catboost.load_model(model_tau_0_uri)
    
    model_tau_1_uri = f"models:/model_tau_1/{version}"
    print(f"Загрузка model_tau_1 из реестра: {model_tau_1_uri}")
    model_tau_1 = mlflow.catboost.load_model(model_tau_1_uri)
    
    # Загружаем калиброванный классификатор propensity
    model_propensity_uri = f"models:/model_propensity/{version}"
    print(f"Загрузка model_propensity из реестра: {model_propensity_uri}")
    model_propensity = mlflow.sklearn.load_model(model_propensity_uri)
    
    print("Компоненты успешно извлечены из S3")
    return S3XLearnerPredictor(model_tau_0, model_tau_1, model_propensity)

def sanitize_features_for_catboost(X, cat_features):
    X_out = X.copy()
    for col in cat_features:
        if col in X_out.columns:
            X_out[col] = X_out[col].astype('object')
            X_out[col] = X_out[col].where(X_out[col].notna(), '__nan__')
            X_out[col] = X_out[col].astype(str)
    return X_out

def main():
    # Имитируем сырой входящий JSON-запрос от клиента (ниже дан список необходимых признаков)
    mock_payload = {
        "client": [
            {
                "client_id": 999,
                "age": 34.0,
                "gender": "F",
                "first_issue_date": "2017-09-01 10:00:00",
                "first_redeem_date": "2017-11-20 18:00:00"
            }
        ],
        "purchases": [
            {
                "client_id": 999,
                "transaction_id": "tx_demo_1",
                "transaction_datetime": "2018-10-05 12:00:00",
                "regular_points_received": 12.0,
                "express_points_received": 0.0,
                "regular_points_spent": 0.0,
                "express_points_spent": 0.0,
                "purchase_sum": 600.0,
                "store_id": "store_101",
                "product_id": "prod_x",
                "product_quantity": 1.0,
                "trn_sum_from_iss": 600.0,
                "trn_sum_from_red": 600.0
            },
            {
                "client_id": 999,
                "transaction_id": "tx_demo_2",
                "transaction_datetime": "2018-12-15 17:30:00",
                "regular_points_received": 20.0,
                "express_points_received": 0.0,
                "regular_points_spent": 10.0,
                "express_points_spent": 0.0,
                "purchase_sum": 1500.0,
                "store_id": "store_101",
                "product_id": "prod_y",
                "product_quantity": 2.0,
                "trn_sum_from_iss": 1400.0,
                "trn_sum_from_red": 1400.0
            }
        ]
    }

    try:
        predictor = load_models_from_s3_registry(version=2) # версия PRD модели на момент прогона ноутбука
    except Exception as e:
        print(f"\nНе удалось загрузить модели из реестра S3: {e}")
        return

    # Предобработка сырого запроса
    print("\nЗапуск UpliftFeatureExtractorExpanded")
    client_df = pd.DataFrame(mock_payload["client"])
    purchases_df = pd.DataFrame(mock_payload["purchases"])
    
    fe = UpliftFeatureExtractorExpanded(drop_redundant=True)
    
    # Создаем фиктивные датафреймы со значениями по умолчанию для совместимости с интерфейсом экстрактора
    train_stub = pd.DataFrame({"client_id": [999]})
    treatment_stub = pd.Series([0], name="treatment_flg")
    target_stub = pd.Series([0], name="target")
    
    features_df = fe.calculate_features(
        clients_df=client_df,
        train_df=train_stub,
        treatment_df=treatment_stub,
        target_df=target_stub,
        purchases_df=purchases_df
    )
    
    # Убираем вспомогательные таргеты обучения
    X_plain = features_df.drop(columns=["treatment_flg", "target"], errors="ignore")

    print("\nПрименение UMAP + KMeans")

    try:
        clusterizer = joblib.load("clusterizer.joblib")
    except FileNotFoundError:
        print("\nФайл не найден в текущей директории.")
        return
        
    X_clustered = clusterizer.transform(X_plain)
    
    cat_cols = X_clustered.select_dtypes(include=['object', 'category']).columns.tolist()
    X_final = sanitize_features_for_catboost(X_clustered, cat_cols)

    print("\nРасчет Uplift")
    uplift = predictor.predict_uplift(X_final)
    
    print("=" * 60)
    print(f"Результат для клиента {mock_payload['client'][0]['client_id']}:")
    print(f"Предсказанный Uplift-эффект: {uplift[0]:.6f}")

if __name__ == "__main__":
    main()


Подключение к S3 и загрузка компонентов модели
Загрузка model_tau_0 из реестра: models:/model_tau_0/2


Загрузка model_tau_1 из реестра: models:/model_tau_1/2


Загрузка model_propensity из реестра: models:/model_propensity/2


Компоненты успешно извлечены из S3

Запуск UpliftFeatureExtractorExpanded

Применение UMAP + KMeans

Расчет Uplift
Результат для клиента 999:
Предсказанный Uplift-эффект: 0.031670
